# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [ ]:
!pip install llama-index llama-index-llms-openai-like llama-index-embeddings-openai python-dotenv spacy

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).parent))
from config import get_llm, get_embeddings, LLM_CONFIG, EMBEDDING_CONFIG, check_connections

In [ ]:
# Verify config and test connections
check_connections()

In [ ]:
import nest_asyncio
import random
import time

nest_asyncio.apply()

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.prompts import PromptTemplate
from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator
)
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai import OpenAIEmbedding

### Configure LlamaIndex to use custom LLM & Embedding endpoints

In [ ]:
# Configure LlamaIndex Settings to use our custom endpoints
Settings.llm = OpenAILike(
    model=LLM_CONFIG["model"],
    api_base=LLM_CONFIG["base_url"],
    api_key=LLM_CONFIG["api_key"],
    temperature=0,
    is_chat_model=True,
    context_window=32000,
    timeout=600,
)

Settings.embed_model = OpenAIEmbedding(
    model_name=EMBEDDING_CONFIG["model"],
    api_base=EMBEDDING_CONFIG["base_url"],
    api_key=EMBEDDING_CONFIG["api_key"],
    timeout=120,
)

print(f"LLM: {LLM_CONFIG['model']} @ {LLM_CONFIG['base_url']}")
print(f"Embedding: {EMBEDDING_CONFIG['model']} @ {EMBEDDING_CONFIG['base_url']}")

### Read Docs

In [ ]:
data_dir = "../data"
documents = SimpleDirectoryReader(data_dir).load_data()

### Create evaluation questions and pick k out of them

In [ ]:
num_eval_questions = 25

eval_documents = documents[0:20]
data_generator = DatasetGenerator.from_documents(eval_documents)
eval_questions = data_generator.generate_questions_from_nodes()
k_eval_questions = random.sample(eval_questions, num_eval_questions)

### Define metrics evaluators

In [ ]:
# Define Faithfulness Evaluator
faithfulness_evaluator = FaithfulnessEvaluator()

faithfulness_new_prompt_template = PromptTemplate("""Please tell if a given piece of information is directly supported by the context.
You need to answer with either YES or NO.
Answer YES if any part of the context explicitly supports the information, even if most of the context is unrelated.
If the context does not explicitly support the information, answer NO.

Information: Apple pie is generally double-crusted.
Context: An apple pie is a fruit pie in which the principal filling ingredient is apples.
Apple pie is often served with whipped cream, ice cream ('apple pie à la mode'), custard, or cheddar cheese.
It is generally double-crusted, with pastry both above and below the filling; the upper crust may be solid or latticed.
Answer: YES

Information: Apple pies taste bad.
Context: An apple pie is a fruit pie in which the principal filling ingredient is apples.
Apple pie is often served with whipped cream, ice cream ('apple pie à la mode'), custard, or cheddar cheese.
Answer: NO

Information: {query_str}
Context: {context_str}
Answer:
""")

faithfulness_evaluator.update_prompts({"your_prompt_key": faithfulness_new_prompt_template})

# Define Relevancy Evaluator
relevancy_evaluator = RelevancyEvaluator()

### Function to evaluate metrics for each chunk size

In [ ]:
def evaluate_response_time_and_accuracy(chunk_size, eval_questions):
    """
    Evaluate the average response time, faithfulness, and relevancy
    for a given chunk size.
    """
    total_response_time = 0
    total_faithfulness = 0
    total_relevancy = 0

    Settings.chunk_size = chunk_size
    Settings.chunk_overlap = chunk_size // 5

    vector_index = VectorStoreIndex.from_documents(eval_documents)
    query_engine = vector_index.as_query_engine(similarity_top_k=5)
    num_questions = len(eval_questions)

    for question in eval_questions:
        start_time = time.time()
        response_vector = query_engine.query(question)
        elapsed_time = time.time() - start_time

        faithfulness_result = faithfulness_evaluator.evaluate_response(
            response=response_vector
        ).passing

        relevancy_result = relevancy_evaluator.evaluate_response(
            query=question, response=response_vector
        ).passing

        total_response_time += elapsed_time
        total_faithfulness += faithfulness_result
        total_relevancy += relevancy_result

    average_response_time = total_response_time / num_questions
    average_faithfulness = total_faithfulness / num_questions
    average_relevancy = total_relevancy / num_questions

    return average_response_time, average_faithfulness, average_relevancy

### Test different chunk sizes

In [ ]:
chunk_sizes = [128, 256]

for chunk_size in chunk_sizes:
    avg_response_time, avg_faithfulness, avg_relevancy = evaluate_response_time_and_accuracy(chunk_size, k_eval_questions)
    print(f"Chunk size {chunk_size} - Average Response time: {avg_response_time:.2f}s, Average Faithfulness: {avg_faithfulness:.2f}, Average Relevancy: {avg_relevancy:.2f}")